[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jsonmen/bias-and-variance/blob/main/nlp/SentimentAnalysis/IMDB50K/CustomTextPreprocessing.ipynb)

# Abstract

- Goal: Make a good text preprocessing pipeline

- Dataset: [IMDB Dataset of 50K Movie Reviews (Kaggle)](https://www.kaggle.com/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews)

- Project Details:

    I'm just trying to learn how to use text preprocessing tools and how to set up my own text preprocessing pipeline. Also, I learned different text preprocessing techniques

- Best result: This function help to achieve 0.839 (Accuracy Score of Logistic Regression) without this function i have ~0.64 (Accuracy Score of Logistic Regression)
- Sections:

    - [Install Dependecies](#Install-Dependecies)
    - [Imports](#Imports)
    - [Dataset](#Dataset)
    - [Text Preprocessing](#Text-Preprocessing)
    - [Test of text preprocessing function](#Test-of-text-preprocessing-function)

# Download & Install Dependencies

In [ ]:
# Install modules (if needed)
!pip install emoji symspellpy spacy pandas nltk

!python -m spacy download en_core_web_sm

In [ ]:
# Dataset Downloading
!mkdir data
!curl -L -o ./data/dataset.zip https://www.kaggle.com/api/v1/datasets/download/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews
!unzip ./data/dataset.zip -d ./data
!rm ./data/dataset.zip
!mv ./data/IMDB\ Dataset.csv ./data/imdb_dataset.csv

In [ ]:
# For colab users (local library files download)
!curl -L -o ./setup_text_preprocessing.py https://raw.githubusercontent.com/jsonmen/bias-and-variance/refs/heads/main/SentimentAnalysis/setup_text_preprocessing.py

# Imports

In [2]:
import nltk
nltk.download('words')

import re
import pandas as pd
import emoji as e
import spacy
from nltk.corpus import words
from symspellpy import Verbosity
from setup_text_preprocessing import load_spellcorrector, HTML_CLEANER, URL_CLEANER, SLANG_MAP, EMOTICON_MAP, CORRECTION_WHITELIST # Config file for text preprocessing

[nltk_data] Downloading package words to /home/ilya-prg/nltk_data...
[nltk_data]   Package words is already up-to-date!


# Dataset

In [3]:
dataset = pd.read_csv("./data/imdb_dataset.csv")

In [4]:
dataset.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


# Text Preprocessing

In [5]:
nlp = spacy.load("en_core_web_sm")
sym_spell = load_spellcorrector()

In [6]:
def replace_words_with_dict(text, dictionary):
    """Replace words in text using a dictionary, prioritizing longer phrases."""
    sorted_words = sorted(dictionary.keys(), key=len, reverse=True)
    for word in sorted_words:
        pattern = r'\b' + re.escape(word) + r'\b'
        text = re.sub(pattern, dictionary[word], text, flags=re.IGNORECASE)
    return text.lower()

In [7]:
def text_correction(text, sym_spell=sym_spell, whitelist=CORRECTION_WHITELIST):
    """Autocorrect words not in the whitelist using SymSpell."""
    words = text.split()
    corrected_words = []
    for word in words:
        if word in whitelist:
            corrected_words.append(word)
        else:
            suggestions = sym_spell.lookup(word, Verbosity.CLOSEST, max_edit_distance=2)
            corrected_words.append(suggestions[0].term if suggestions else word)
    return " ".join(corrected_words)

In [8]:
def text_lemmatization(text):
    """Lemmatize text using spaCy."""
    doc = nlp(text)
    return " ".join([token.lemma_ for token in doc])

In [9]:
def text_preprocessing(text):
    """
    Preprocess text for NLP tasks (e.g., sentiment analysis).
    Steps are ordered to preserve meaning and optimize efficiency.
    """
    # 1. Remove HTML tags
    cleaned_text = re.sub(HTML_CLEANER, '', text)
    # 2. Replace hyphens with spaces
    cleaned_text = cleaned_text.replace('-', ' ') 
    # 3. Remove URLs
    cleaned_text = re.sub(URL_CLEANER, '', cleaned_text)
    # 4. Convert to lowercase for consistency
    cleaned_text = cleaned_text.lower()
    # 5. Convert emojis to text and make readable (replace ':' and '_' with spaces)
    cleaned_text = e.demojize(cleaned_text)
    cleaned_text = cleaned_text.replace(':', ' ').replace('_', ' ')
    # 6. Convert emoticons to text
    cleaned_text = replace_words_with_dict(cleaned_text, EMOTICON_MAP)
    # 7. Replace slang with full phrases
    cleaned_text = replace_words_with_dict(cleaned_text, SLANG_MAP)
    # 8. Remove numbers
    cleaned_text = re.sub(r'\d+', '', cleaned_text)
    # 9. Autocorrect misspelled words
    cleaned_text = text_correction(cleaned_text)
    # 10. Remove non-alphabetic characters except spaces
    cleaned_text = re.sub(r'[^a-zA-Z\s]', '', cleaned_text)
    # 11. Normalize whitespace (replace multiple spaces with one)
    cleaned_text = re.sub(r'\s+', ' ', cleaned_text)
    # 12. Strip leading/trailing whitespace
    cleaned_text = cleaned_text.strip()
    # 13. Lemmatize to reduce words to base forms
    cleaned_text = text_lemmatization(cleaned_text)
    return cleaned_text

# Test of text preprocessing function

In [10]:
%%time
for i in range(2):
    text = dataset.iloc[i]["review"]
    print(f"{text}\n ↓ \n{text_preprocessing(text)}\n\n")

One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<br /><br />The first thing that struck me about Oz was its brutality and unflinching scenes of violence, which set in right from the word GO. Trust me, this is not a show for the faint hearted or timid. This show pulls no punches with regards to drugs, sex or violence. Its is hardcore, in the classic use of the word.<br /><br />It is called OZ as that is the nickname given to the Oswald Maximum Security State Penitentary. It focuses mainly on Emerald City, an experimental section of the prison where all the cells have glass fronts and face inwards, so privacy is not high on the agenda. Em City is home to many..Aryans, Muslims, gangstas, Latinos, Christians, Italians, Irish and more....so scuffles, death stares, dodgy dealings and shady agreements are never far away.<br /><br />I would say the main appeal of the show is due to the fac